In [1]:
from docx import Document
from pathlib import Path
import os

In [4]:
# Step 1: Provide the absolute path to the DOCX file
doc_path = "/Users/dulin/Dropbox/UTHealth/SAFElab/LLM/UTH_IPSS_Pediatric Stroke/Instructor/Deidentified_Patients.docx"

# Step 2: Check if the file exists
if os.path.exists(doc_path):
    # Step 3: Load the DOCX file
    document = Document(doc_path)
    
    # Step 4: Extract and print all paragraphs from the document
    text = "\n".join([para.text for para in document.paragraphs if para.text.strip()])
    
    print("✅ Document loaded successfully! Here’s the extracted text preview:")
    print(text[:500])  # Print first 500 characters as a preview
else:
    print("❌ File not found. Please check the file path and try again.")


✅ Document loaded successfully! Here’s the extracted text preview:
30130
Dx: Perinatal Arterial Ischemic Stroke
Patient is a 2 y.o. male referred for an initial evaluation in pediatric stroke clinic at the request of Physician because of a history of perinatal arterial ischemic strokes.  History is per mother and review of medical records made available.
Patient was born at Hospital via stat c-section for failure to progress, but had APGARS of 8 and 8 and did not require cooling. Shortly after birth he was noted to have left sided focal motor movements and was 


In [5]:
# Variables to store notes and case numbers
case_notes = {}
current_case_number = None
case_count = {}

# Extract notes considering page breaks
for paragraph in document.paragraphs:
    text = paragraph.text.strip()
    
    # Check if paragraph has any runs before looking for page breaks
    if paragraph.runs and paragraph.runs[0].element.xml.find('<w:br w:type="page"/>') > 0:
        current_case_number = None  # Reset current case number on new page
    
    # Check if this is a new case number based on the first line of the note
    if not current_case_number and text and text[0].isdigit():
        current_case_number = text.split()[0]  # Only take the first number part
        
        # Increment the count for this case number
        if current_case_number in case_count:
            case_count[current_case_number] += 1
        else:
            case_count[current_case_number] = 1
            
        # Append visit count to case number
        current_case_number_with_visit = f"{current_case_number}-{case_count[current_case_number]}"
        case_notes[current_case_number_with_visit] = []
    
    if current_case_number:
        case_notes[current_case_number_with_visit].append(text)

# Save each case note into a separate text file with the modified case numbers
output_folder = Path("notes/")
output_folder.mkdir(parents=True, exist_ok=True)

for case_number, notes in case_notes.items():
    if notes:
        filename = output_folder / f"{case_number}.txt"
        with open(filename, 'w') as file:
            file.write('\n'.join(notes))

print(f"Files saved in directory: {output_folder}")

Files saved in directory: notes


In [6]:
len(os.listdir("notes"))

50